In [ ]:
!pip install sentence_transformers
!pip install torch_geometric

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\jaden\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# load json
import json
from sentence_transformers import SentenceTransformer
import json
import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from torch_geometric.data import Data, InMemoryDataset
print("import ok")

import ok


In [ ]:
with open("wiki_data_yearly_triples.json", 'r') as f:
    data = json.load(f)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5186.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# get global node set
global_nodes = set()

for key in data.keys():
    for triple in data[key]:
        global_nodes.add(triple["head"])
        global_nodes.add(triple["tail"])
        
global_nodes = list(global_nodes)
print(len(global_nodes))

10834


In [ ]:
# get global_rel set
global_rel = set()

for key in data.keys():
    for triple in data[key]:
        global_rel.add(triple["relation"])
        
global_rel = list(global_rel)
print(len(global_rel))

23


In [ ]:
# index mappings for nodes and rels
id_to_node = {}
id_to_rel = {}

node_to_id = {}
rel_to_id = {}

for i, node in enumerate(global_nodes):
    id_to_node[i] = node
    node_to_id[node] = i

for i, rel in enumerate(global_rel):
    id_to_rel[i] = rel
    rel_to_id[rel] = i

In [ ]:
def embed(texts, batch_size=64):
    # this will become more advanced as we encorperate more and more into the embedding
    # EX: emb = name_emb + description_emb + class_emb
    #    - or concat for more representation
    embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embs.astype(np.float32)

In [ ]:
from wikidata_dataset_class import GlobalTemporalTextKGDataset

In [ ]:
# for classes I can just add a id_to_class mapping in the dataset class
dataset = GlobalTemporalTextKGDataset(
    root="data/wikidata",
    start_year=1800,
    json_filename="wikidata_with_descriptions.json",
    dataset_filename="descriptions_dataset_1800.pt",
    entity_to_id=node_to_id,
    relation_to_id=rel_to_id,
    embed_fn=embed,
)